# 3b. COMPASS UMAP Visualization

Визуализация 3D UMAP проекций COMPASS geneset-level embeddings.

**Requires:** Notebook `3.compass_embedding.ipynb` must have been run first,
results stored in `data/interim/`.

**Plots:**
- Per-cohort: 2×3 grid — top row by **RNA Batch**, bottom row by **Diagnosis**
- Combined: all cohorts overlaid, colored by Cohort / Batch / Diagnosis

## 0. Setup

In [ ]:
!git clone https://github.com/onion-42/batchcor-rna-embeds.git 2>/dev/null || (cd batchcor-rna-embeds && git pull origin main)
%cd batchcor-rna-embeds

!pip install -q uv
!uv pip install --system -e .
!pip install -q --upgrade ipython ipykernel

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from pathlib import Path
from loguru import logger

from batchcor_rna_emb.logging_config import set_logger
from batchcor_rna_emb.config import (
    BATCH_COL, DIAGNOSIS_COL,
    COMPASS_UMAP_KEY, COMPASS_EMBEDDING_KEY,
    COMPASS_PCA_KEY, COMPASS_METADATA_KEY,
    COHORT_CANCER_CODE,
)
from batchcor_rna_emb.data_io import load_cohort

set_logger(level="INFO")

In [ ]:
# --- Mount Drive & define paths ---
from google.colab import drive
drive.mount('/content/drive')

DATA_INTERIM_DIR = Path('/content/drive/MyDrive/data/interim')

available = sorted(DATA_INTERIM_DIR.glob('*.adata.zarr'))
logger.info("Available interim cohorts ({}): {}", len(available), [p.name for p in available])

## 1. Plot Helpers

In [ ]:
# --- Plotting style ---
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    "figure.facecolor": "#f8f9fa",
    "axes.facecolor":   "#ffffff",
    "axes.edgecolor":   "#dee2e6",
    "grid.color":       "#e9ecef",
    "grid.linewidth":   0.6,
    "figure.dpi":       120,
})

UMAP_PAIRS  = [(0, 1), (0, 2), (1, 2)]
PAIR_LABELS = [("UMAP-1", "UMAP-2"), ("UMAP-1", "UMAP-3"), ("UMAP-2", "UMAP-3")]


def plot_umap_panel(umap_coords, color_labels, title, ax, palette=None, legend=True):
    """Scatter plot of 2D UMAP projection, colored by labels."""
    unique_labels = sorted(color_labels.unique())
    if palette is None:
        palette = sns.color_palette("husl", n_colors=len(unique_labels))
    color_map = dict(zip(unique_labels, palette))
    colors = [color_map[l] for l in color_labels]

    ax.scatter(
        umap_coords[:, 0], umap_coords[:, 1],
        c=colors, s=12, alpha=0.65, edgecolors="none", rasterized=True,
    )
    ax.set_title(title, fontsize=11, fontweight="bold", pad=8)

    if legend and len(unique_labels) <= 15:
        handles = [
            Line2D([0], [0], marker='o', color='w', markerfacecolor=color_map[l],
                   markersize=7, label=str(l))
            for l in unique_labels
        ]
        ax.legend(
            handles=handles, fontsize=7, loc="best",
            framealpha=0.85, edgecolor="#ccc",
            ncol=1 if len(unique_labels) <= 6 else 2,
        )

## 2. Per-Cohort UMAP Plots

Для каждой когорты:
- **Top row** — 3 UMAP projections colored by `RNA_batch`
- **Bottom row** — 3 UMAP projections colored by `Diagnosis`

In [ ]:
for cohort_name in COHORT_CANCER_CODE:
    interim_path = DATA_INTERIM_DIR / f"{cohort_name}.adata.zarr"
    if not interim_path.exists():
        logger.warning("Skipping (not in interim): {}", cohort_name)
        continue

    adata = load_cohort(interim_path)
    umap = np.asarray(adata.obsm[COMPASS_UMAP_KEY])
    batch = adata.obs[BATCH_COL].astype(str)
    diag  = adata.obs[DIAGNOSIS_COL].astype(str)

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(
        f"COMPASS UMAP — {cohort_name}  ({adata.n_obs} samples)",
        fontsize=14, fontweight="bold", y=0.98,
    )

    for col_idx, ((i, j), (xl, yl)) in enumerate(zip(UMAP_PAIRS, PAIR_LABELS)):
        coords = umap[:, [i, j]]
        # top row — batch
        plot_umap_panel(coords, batch, f"{xl} vs {yl} — RNA Batch", axes[0, col_idx])
        axes[0, col_idx].set_xlabel(xl)
        axes[0, col_idx].set_ylabel(yl)
        # bottom row — diagnosis
        plot_umap_panel(
            coords, diag, f"{xl} vs {yl} — Diagnosis", axes[1, col_idx],
            palette=sns.color_palette("Set2", n_colors=diag.nunique()),
        )
        axes[1, col_idx].set_xlabel(xl)
        axes[1, col_idx].set_ylabel(yl)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()
    del adata

logger.info("Per-cohort UMAP visualization complete.")

## 3. Combined All-Cohorts UMAP Overview

UMAP-1 vs UMAP-2 for all cohorts overlaid, colored by:
1. **Cohort** — inter-cohort structure
2. **RNA Batch** — batch effect across cohorts
3. **Diagnosis** — biological signal preservation

In [ ]:
all_umaps, all_cohort_labels, all_batch_labels, all_diag_labels = [], [], [], []

for cohort_name in COHORT_CANCER_CODE:
    interim_path = DATA_INTERIM_DIR / f"{cohort_name}.adata.zarr"
    if not interim_path.exists():
        continue
    adata = load_cohort(interim_path)
    all_umaps.append(np.asarray(adata.obsm[COMPASS_UMAP_KEY]))
    all_cohort_labels.extend([cohort_name] * adata.n_obs)
    all_batch_labels.extend(adata.obs[BATCH_COL].astype(str).tolist())
    all_diag_labels.extend(adata.obs[DIAGNOSIS_COL].astype(str).tolist())
    del adata

umap_all = np.vstack(all_umaps)
df_meta = pd.DataFrame({
    "cohort":    all_cohort_labels,
    "batch":     all_batch_labels,
    "diagnosis": all_diag_labels,
})

logger.info(
    "Combined dataset: {} samples, {} cohorts",
    len(df_meta), df_meta['cohort'].nunique(),
)

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
fig.suptitle(
    f"COMPASS UMAP — All Cohorts Combined  ({len(df_meta)} samples)",
    fontsize=14, fontweight="bold", y=1.02,
)

color_by = [
    ("cohort",    "Cohort",    sns.color_palette("tab10", n_colors=df_meta['cohort'].nunique())),
    ("batch",     "RNA Batch", None),
    ("diagnosis", "Diagnosis", sns.color_palette("Set2",  n_colors=df_meta['diagnosis'].nunique())),
]

for ax, (col, label, pal) in zip(axes, color_by):
    plot_umap_panel(
        umap_all[:, [0, 1]], df_meta[col],
        f"UMAP-1 vs UMAP-2 — {label}", ax,
        palette=pal,
        legend=(df_meta[col].nunique() <= 15),
    )
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")

plt.tight_layout()
plt.show()

logger.info("Combined UMAP visualization complete.")